# Stage 04 — Data Quality Validation Framework

## Enterprise Data Quality & Governance Project

### Objective

The purpose of this stage is to move from individual Data Quality checks toward a reusable validation framework.

Instead of writing a separate result structure for every rule, the framework organizes:
- Data Quality rules
- Validation logic
- Violation counts
- Valid records
- Quality rates
- PASS / FAIL status

The framework is designed so that the same validation approach can be reused when the dataset is assessed again.


## 1. Load and Prepare the Dataset

In [ ]:
import pandas as pd

df = pd.read_csv("customers.csv")

# Standardize the observations identified in earlier stages
df["income_level"] = df["income_level"].replace({
    "LOW": "Low"
})

df["location"] = df["location"].replace({
    "Chennal": "Chennai"
})

df["signup_date"] = pd.to_datetime(
    df["signup_date"],
    errors="coerce"
)

df.head()


## 2. Define the Data Quality Rules

The rules are stored as structured records so they can be maintained as a reusable rule catalog.


In [ ]:
rules_catalog = [
    {
        "rule_id": "DQ001",
        "dimension": "Completeness",
        "field": "customer_id",
        "description": "customer_id must not contain NULL values",
        "threshold": 100
    },
    {
        "rule_id": "DQ002",
        "dimension": "Completeness",
        "field": "age",
        "description": "age must not contain NULL values",
        "threshold": 100
    },
    {
        "rule_id": "DQ003",
        "dimension": "Completeness",
        "field": "gender",
        "description": "gender must not contain NULL values",
        "threshold": 100
    },
    {
        "rule_id": "DQ004",
        "dimension": "Completeness",
        "field": "location",
        "description": "location must not contain NULL values",
        "threshold": 100
    },
    {
        "rule_id": "DQ005",
        "dimension": "Completeness",
        "field": "income_level",
        "description": "income_level must not contain NULL values",
        "threshold": 100
    },
    {
        "rule_id": "DQ006",
        "dimension": "Completeness",
        "field": "signup_date",
        "description": "signup_date must not contain NULL values",
        "threshold": 100
    },
    {
        "rule_id": "DQ007",
        "dimension": "Uniqueness",
        "field": "customer_id",
        "description": "customer_id must be unique",
        "threshold": 100
    },
    {
        "rule_id": "DQ008",
        "dimension": "Validity",
        "field": "age",
        "description": "age must be between 18 and 69",
        "threshold": 100
    },
    {
        "rule_id": "DQ009",
        "dimension": "Validity",
        "field": "gender",
        "description": "gender must be Male or Female",
        "threshold": 100
    },
    {
        "rule_id": "DQ010",
        "dimension": "Validity",
        "field": "income_level",
        "description": "income_level must be High, Medium, or Low",
        "threshold": 100
    },
    {
        "rule_id": "DQ011",
        "dimension": "Validity",
        "field": "location",
        "description": "location must belong to approved locations",
        "threshold": 100
    },
    {
        "rule_id": "DQ012",
        "dimension": "Validity",
        "field": "signup_date",
        "description": "signup_date must contain valid dates",
        "threshold": 100
    },
    {
        "rule_id": "DQ013",
        "dimension": "Consistency",
        "field": "income_level",
        "description": "income_level must use consistent capitalization",
        "threshold": 100
    },
    {
        "rule_id": "DQ014",
        "dimension": "Consistency",
        "field": "location",
        "description": "location must use standardized names",
        "threshold": 100
    },
    {
        "rule_id": "DQ015",
        "dimension": "Consistency",
        "field": "signup_date",
        "description": "signup_date must use a consistent date representation",
        "threshold": 100
    }
]

rules_catalog_df = pd.DataFrame(rules_catalog)
rules_catalog_df


## 3. Define Validation Logic

The function below centralizes the validation logic.

For each rule, it calculates:
- Total records
- Violations
- Valid records
- Quality rate
- PASS / FAIL status

This makes the assessment reusable instead of manually calculating every result.


In [ ]:
allowed_gender = ["Male", "Female"]
allowed_income = ["High", "Medium", "Low"]
allowed_locations = [
    "Mumbai",
    "Bangalore",
    "Chennai",
    "Hyderabad",
    "Kolkata",
    "Delhi"
]


def validate_data(df, rule):
    field = rule["field"]
    rule_id = rule["rule_id"]

    total_records = len(df)

    if rule_id == "DQ001":
        violations = df[field].isnull().sum()

    elif rule_id == "DQ002":
        violations = df[field].isnull().sum()

    elif rule_id == "DQ003":
        violations = df[field].isnull().sum()

    elif rule_id == "DQ004":
        violations = df[field].isnull().sum()

    elif rule_id == "DQ005":
        violations = df[field].isnull().sum()

    elif rule_id == "DQ006":
        violations = df[field].isnull().sum()

    elif rule_id == "DQ007":
        violations = df[field].duplicated().sum()

    elif rule_id == "DQ008":
        violations = ((df[field] < 18) | (df[field] > 69)).sum()

    elif rule_id == "DQ009":
        violations = (~df[field].isin(allowed_gender)).sum()

    elif rule_id == "DQ010":
        violations = (~df[field].isin(allowed_income)).sum()

    elif rule_id == "DQ011":
        violations = (~df[field].isin(allowed_locations)).sum()

    elif rule_id == "DQ012":
        violations = df[field].isnull().sum()

    elif rule_id == "DQ013":
        violations = (~df[field].isin(allowed_income)).sum()

    elif rule_id == "DQ014":
        violations = (~df[field].isin(allowed_locations)).sum()

    elif rule_id == "DQ015":
        violations = df[field].isnull().sum()

    else:
        raise ValueError(f"Unknown rule: {rule_id}")

    valid_records = total_records - violations
    quality_rate = (valid_records / total_records) * 100

    status = "PASS" if quality_rate >= rule["threshold"] else "FAIL"

    return {
        "rule_id": rule_id,
        "dimension": rule["dimension"],
        "field": field,
        "total_records": total_records,
        "violations": violations,
        "valid_records": valid_records,
        "quality_rate": quality_rate,
        "status": status
    }


## 4. Run the Validation Framework

The same validation function is now applied to all rules in the rule catalog.


In [ ]:
validation_results = []

for rule in rules_catalog:
    result = validate_data(df, rule)
    validation_results.append(result)

quality_results = pd.DataFrame(validation_results)

quality_results


## 5. Review Failed Rules

This view is useful in a real monitoring process because it immediately highlights rules that require attention.


In [ ]:
failed_rules = quality_results[
    quality_results["status"] == "FAIL"
]

failed_rules


## 6. Generate the Final Validation Report

In [ ]:
total_rules = len(quality_results)
total_violations = quality_results["violations"].sum()
valid_records = quality_results["valid_records"].sum()

overall_quality_rate = (
    (quality_results["quality_rate"].sum() / total_rules)
)

overall_status = "PASS" if total_violations == 0 else "FAIL"

final_validation_report = {
    "total_records": len(df),
    "total_rules": total_rules,
    "total_violations": total_violations,
    "valid_records_across_rules": valid_records,
    "quality_rate": overall_quality_rate,
    "status": overall_status
}

final_validation_report


## 7. Validation Result

For the current standardized dataset, the framework is expected to reproduce the validated results established in the previous stages:

- **15 rules**
- **0 violations**
- **100% quality rate**
- **PASS**

The important improvement in this stage is not the numerical result itself. It is that the result is now generated through a reusable validation function and structured rule catalog.


## 8. Framework Flow

The validation process can be represented as:

**Dataset → Rule Catalog → Validation Logic → Violation Count → Quality Rate → PASS / FAIL → Validation Report**

This structure can later be extended for:
- Additional datasets
- Additional Data Quality rules
- Quality thresholds
- Automated monitoring
- Historical quality tracking


# Stage 04 Conclusion

Stage 04 transformed the Data Quality assessment from a collection of individual checks into a reusable validation framework.

The framework uses a structured rule catalog and a `validate_data()` function to apply the rules consistently and produce standardized validation results.

For the current dataset, the framework reproduces the project result of **15 rules, 0 violations, 100% quality rate, and PASS**.

This provides the technical foundation for future monitoring, where the same rules can be executed repeatedly and the results can be compared over time.
